## Import Python Libraries and load environment variables

In [35]:
import os
import json
import urllib.request
import urllib.error
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

load_dotenv()

True

In [36]:
llm = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    timeout=300,
    temperature=0.5,
    max_tokens=25000
)

## Create a helper function to beautify llm responses in our Jupyter Notebook

In [37]:
def bfmsgs(result):
    """Extract and display text content from agent result messages."""
    text = "\n\n".join(
        block["text"] for block in result["messages"][-1].content_blocks
        if block.get("type") == "text"
    )
    display(Markdown(text))

In [38]:
# bfmsgs(result)

## Create a dummy function that returns 1500 as KRW > USD FX

In [39]:
def get_fx(currency: str) -> float:
    """Return conversion rate from KRW to the given currency (e.g., USD)."""
    # Read API key from environment variables loaded by load_dotenv()
    api_key = os.getenv("EXCHANGE_RATE_API_KEY")
    if not api_key:
        raise RuntimeError("Missing EXCHANGE_RATE_API_KEY in .env")

    # Build ExchangeRate-API endpoint with KRW as base currency
    url = f"https://v6.exchangerate-api.com/v6/{api_key}/latest/KRW"

    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; hai5016-project/1.0)"},
    )

    try:
        with urllib.request.urlopen(req, timeout=30) as response:
            data = json.loads(response.read().decode("utf-8"))
    except urllib.error.URLError as error:
        raise RuntimeError(f"Failed to fetch exchange rates: {error}") from error

    # Optional API-level error check
    if data.get("result") != "success":
        error_type = data.get("error-type", "unknown_error")
        raise RuntimeError(f"ExchangeRate API error: {error_type}")

    rates = data.get("conversion_rates", {})
    code = currency.upper()

    if code not in rates:
        raise ValueError(f"Currency code not found: {code}")

    return float(rates[code])

In [40]:
from exchangerates import get_krw_conversions


print(get_krw_conversions())

2026-06-03 16:32:30.435 | INFO     | exchangerates:_get_cached_rates_via_postgrest:191 - Checking Supabase cache via PostgREST
2026-06-03 16:32:31.033 | INFO     | exchangerates:get_krw_conversions:76 - Using cached FX rates from Supabase (166 rows)


{'KRW': 1.0, 'AED': 0.002421, 'AFN': 0.04147, 'ALL': 0.05399, 'AMD': 0.2427, 'ANG': 0.00118, 'AOA': 0.6206, 'ARS': 0.9407, 'AUD': 0.00091879, 'AWG': 0.00118, 'AZN': 0.00112, 'BAM': 0.001108, 'BBD': 0.001319, 'BDT': 0.08093, 'BGN': 0.001108, 'BHD': 0.00024788, 'BIF': 1.9612, 'BMD': 0.00065925, 'BND': 0.00084296, 'BOB': 0.004557, 'BRL': 0.003309, 'BSD': 0.00065925, 'BTN': 0.06288, 'BWP': 0.009064, 'BYN': 0.001817, 'BZD': 0.001319, 'CAD': 0.00091207, 'CDF': 1.5075, 'CHF': 0.00051842, 'CLF': 1.487e-05, 'CLP': 0.588, 'CNH': 0.004458, 'CNY': 0.004485, 'COP': 2.3478, 'CRC': 0.2999, 'CUP': 0.01582, 'CVE': 0.06245, 'CZK': 0.01374, 'DJF': 0.1172, 'DKK': 0.004225, 'DOP': 0.03847, 'DZD': 0.08758, 'EGP': 0.0342, 'ERN': 0.009889, 'ETB': 0.1043, 'EUR': 0.00056636, 'FJD': 0.001448, 'FKP': 0.00048953, 'FOK': 0.004225, 'GBP': 0.00048953, 'GEL': 0.001754, 'GGP': 0.00048953, 'GHS': 0.007767, 'GIP': 0.00048953, 'GMD': 0.04891, 'GNF': 5.7777, 'GTQ': 0.005022, 'GYD': 0.1377, 'HKD': 0.005166, 'HNL': 0.01752, 

In [41]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
    tools=[get_fx]
)

In [42]:
from datetime import datetime
import re

@tool
def fetch_text_from_url(url: str) -> str:
    """Fetch menu-like text blocks from a URL and keep only useful content."""
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )

    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as error:
        return f"Fetch failed: {error}"

    # Decode HTML safely
    html = raw.decode("utf-8", errors="replace")
    soup = BeautifulSoup(html, "html.parser")

    # Remove non-content tags that add a lot of noise
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav"]):
        tag.decompose()

    # Keep likely menu containers first (table/list/sections with diet/menu keywords)
    candidate_blocks = []
    keyword_pattern = re.compile(r"(식단|메뉴|중식|석식|조식|lunch|dinner|breakfast|menu)", re.IGNORECASE)

    selectors = [
        "table",
        "ul",
        "ol",
        "section",
        "article",
        "div",
    ]

    for selector in selectors:
        for node in soup.select(selector):
            text = node.get_text(" ", strip=True)
            if len(text) < 30:
                continue
            if keyword_pattern.search(text):
                candidate_blocks.append(text)

    # Fallback to body text if no candidate block is found
    if not candidate_blocks and soup.body:
        candidate_blocks = [soup.body.get_text(" ", strip=True)]

    # De-duplicate and keep output small enough for the model
    unique_blocks = []
    seen = set()
    for block in candidate_blocks:
        normalized = " ".join(block.split())
        if normalized not in seen:
            seen.add(normalized)
            unique_blocks.append(normalized)

    # Include today's date to improve downstream filtering
    today = datetime.now()
    today_hint = (
        f"TODAY_HINT: {today.strftime('%Y-%m-%d')} / "
        f"{today.strftime('%m/%d')} / "
        f"{today.strftime('%-m/%-d')}"
    )

    trimmed_text = "\n\n".join(unique_blocks[:20])
    return f"SOURCE_URL: {url}\n{today_hint}\n\n{trimmed_text[:25000]}"

In [43]:
language = "English"
SYSTEM_PROMPT = f"""You are a menu finder assistant. You can find menu information and prices from restaurant websites in Korea and provide it to users. Always answer in {language} and use the following tools below to fetch menu information.

## Capabilities

- `fetch_text_from_url`: loads website text from a URL into the conversation. Be aware of the structure of a website and extract only relevant information."""

In [44]:
agent = create_agent(
    model=llm,
    tools=[fetch_text_from_url, get_fx],
    system_prompt=SYSTEM_PROMPT
)

- https://www.hanyang.ac.kr/re12
- http://hfspn.co/

In [45]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's on the menu today at university campuses? Find it at https://www.hanyang.ac.kr/re12 and http://hfspn.co/. After you find it covert your answer to HTML so that it can ben send via email."}]}
)
bfmsgs(result)

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <title>Campus Menu Today</title>
  <style>
    body { font-family: Arial, sans-serif; line-height: 1.5; color: #333; }
    h1 { font-size: 28px; margin-bottom: 12px; }
    h2 { font-size: 22px; color: #2c3e50; margin-top: 24px; }
    h3 { font-size: 18px; color: #34495e; margin-top: 16px; }
    ul { margin: 0 0 12px 20px; }
    .note { font-size: 12px; color: #555; }
    .campus { margin-bottom: 24px; }
    .price { font-weight: bold; }
  </style>
</head>
<body>
  <h1>Today’s Campus Menu</h1>

  <div class="campus" aria-label="HY-in Hanyang University Portal - Seoul & ERICA Campuses">
    <h2>HY-in Portal Menu (Seoul Campus) and ERICA Campus</h2>

    <h3>Today’s Menu (Seoul Campus)</h3>
    <p class="note">Note: The HY-in campus dining halls list “오늘의 메뉴” (Today’s Menu) and menus can change. Prices are in KRW.</p>
    <ul>
      <li>
        Breakfast (조식) - 1,000 KRW
        <ul>
          <li>닭고기김치볶음 (Stir-fried Chicken & Kimchi)</li>
          <li>쌀밥</li>
          <li>옥수수전</li>
          <li>도시락김</li>
          <li>간장고추지</li>
          <li>깍두기</li>
        </ul>
      </li>
      <li>
        Lunch (중식) - 4,500 KRW
        <ul>
          <li>훈제오리블랙빈볶음밥 (Smoked Duck Fried Rice with Black Bean Sauce)</li>
          <li>고구마맛탕</li>
          <li>배추김치</li>
          <li>요구르트</li>
        </ul>
      </li>
      <li>
        Lunch (중식) - 6,500 KRW
        <ul>
          <li>비빔냉면&간장불고기 (Spicy Buckwheat Noodles with Soy Sauce-marinated Beef)</li>
          <li>김가루양념밥</li>
          <li>군만두튀김</li>
          <li>무초절임</li>
          <li>요구르트</li>
        </ul>
      </li>
    </ul>

    <h3>Today’s Menu (ERICA Campus)</h3>
    <p class="note">Note: ERICA campus shows the same TODAY’s MENU as listed on HY-in portal sections for today. Prices are in KRW.</p>
    <ul>
      <li>
        Breakfast (조식) - 1,000 KRW
        <ul>
          <li>닭고기김치볶음 (Stir-fried Chicken & Kimchi)</li>
          <li>쌀밥</li>
          <li>옥수수전</li>
          <li>도시락김</li>
          <li>간장고추지</li>
          <li>깍두기</li>
        </ul>
      </li>
      <li>
        Lunch (중식) - 4,500 KRW
        <ul>
          <li>훈제오리블랙빈볶음밥 (Smoked Duck Fried Rice with Black Bean Sauce)</li>
          <li>고구마맛탕</li>
          <li>배추김치</li>
          <li>요구르트</li>
        </ul>
      </li>
      <li>
        Lunch (중식) - 6,500 KRW
        <ul>
          <li>비빔냉면&간장불고기 (Spicy Buckwheat Noodles with Soy Sauce-marinated Beef)</li>
          <li>김가루양념밥</li>
          <li>군만두튀김</li>
          <li>무초절임</li>
          <li>요구르트</li>
        </ul>
      </li>
    </ul>

    <p class="note">Source: HY-in Portal and related campus dining pages. Today’s menu subject to change.</p>
  </div>

  <div class="campus" aria-label="HFSPN Campus Cafeteria">
    <h2>HFSPN Campus Cafeteria - Today’s Menu</h2>
    <p class="note">Breakfast and meals are listed with times and prices in KRW. Some items may have student pricing.</p>

    <ul>
      <li>
        Breakfast 08:00–10:00 – 4,000 KRW
        <ul>
          <li>Tuna Kimchi Jigae (Kimchi Stew)</li>
          <li>Seasoned Veggie Salad</li>
          <li>Potato Croquet</li>
          <li>Radish Kimchi</li>
          <li>밀감푸딩 (Mandarin Pudding)</li>
          <li>학생 1,000 (Student Price: 1,000 KRW)</li>
        </ul>
      </li>

      <li>
        Lunch 1 11:00–14:30 – 4,000 KRW
        <ul>
          <li>Pork Loin Cutlet</li>
          <li>Cream Soup</li>
          <li>Leaf Wraps</li>
          <li>Salad</li>
          <li>Kimchi</li>
        </ul>
      </li>

      <li>
        Lunch 2 11:00–14:30 – 3,500 KRW
        <ul>
          <li>Vegetable Bibimbap</li>
          <li>Tofu Soybean Soup</li>
          <li>Fried Eggs</li>
          <li>Stir Fried Eggplant And Paprika</li>
        </ul>
      </li>

      <li>
        Lunch Noodles 11:00–14:00 – 3,000 KRW
        <ul>
          <li>Janchi Guksu (Anchovy Based Noodle Soup)</li>
        </ul>
      </li>

      <li>
        Dinner 16:40–18:40 – 4,000 KRW
        <ul>
          <li>Spicy Grilled Chicken With Rice</li>
          <li>Korean Beef Stew</li>
          <li>Cold Platters With Crab Kimchi, Fruits</li>
          <li>Chicken</li>
        </ul>
      </li>

      <li>
        Snack 09:00–18:40
        <ul>
          <li>Ramen With Ricecake Balls (2,400 KRW)</li>
          <li>Cheese Ramen (2,400 KRW)</li>
          <li>치즈토스트 (Cheese Toast) (2,100 KRW)</li>
          <li>Toast (1,800 KRW)</li>
          <li>Ramen (2,100 KRW)</li>
          <li>Bowl Of Rice (700 KRW)</li>
          <li>Cheese (400 KRW)</li>
          <li>김밥은판매하지않습니다. (Kimbap not sold)</li>
        </ul>
      </li>

      <li>
        Lunch 11:00–13:30 – 6,500 KRW
        <ul>
          <li>소불고기버섯컵밥</li>
          <li>건새우아욱된장국</li>
          <li>Potato Salad</li>
          <li>참나물흑임자생채</li>
          <li>Radish Kimchi</li>
          <li>슈퍼볼 쿠키앤크림</li>
        </ul>
      </li>
    </ul>

    <p class="note">Prices and items are indicative and may change. Reference page shows today’s menu sections and daily variations.</p>
  </div>

  <div class="campus" aria-label="Source URLs">
    <p class="note" style="margin-top:8px;">
      Source URLs:
      <a href="https://www.hanyang.ac.kr/re12">https://www.hanyang.ac.kr/re12</a> •
      <a href="http://hfspn.co/">http://hfspn.co/</a>
    </p>
  </div>
</body>
</html>

In [46]:
def send_agent_response_via_smtp2go(agent_result: dict, subject: str = "HAI5016 Meal Finder Update") -> dict:
    """Send the latest agent response text to SMTP2GO recipient email."""
    logger.info("Starting SMTP2GO send process")

    # Read SMTP2GO settings from environment variables (.env already loaded earlier)
    api_key = os.getenv("SMTP2GO_API_KEY")
    sender_email = os.getenv("SMTP2GO_SENDER_EMAIL")
    recipient_email = os.getenv("SMTP2GO_RECIPIENT_EMAIL")

    # Validate required environment variables
    if not api_key:
        logger.error("Missing SMTP2GO_API_KEY in environment")
        raise RuntimeError("Missing SMTP2GO_API_KEY in .env")
    if not sender_email:
        logger.error("Missing SMTP2GO_SENDER_EMAIL in environment")
        raise RuntimeError("Missing SMTP2GO_SENDER_EMAIL in .env")
    if not recipient_email:
        logger.error("Missing SMTP2GO_RECIPIENT_EMAIL in environment")
        raise RuntimeError("Missing SMTP2GO_RECIPIENT_EMAIL in .env")

    logger.info("SMTP2GO environment variables loaded successfully")

    # Extract latest AI response text from agent result
    messages = agent_result.get("messages", [])
    if not messages:
        logger.error("agent_result has no messages")
        raise ValueError("No messages found in agent_result")

    latest_text = None
    for message in reversed(messages):
        if getattr(message, "type", "") == "ai":
            content = getattr(message, "content", "")
            if isinstance(content, str):
                latest_text = content
            elif isinstance(content, list):
                # Handle structured content blocks if returned as a list
                latest_text = "\n".join(
                    str(block.get("text", "")) if isinstance(block, dict) else str(block)
                    for block in content
                )
            else:
                latest_text = str(content)
            break

    if not latest_text:
        logger.error("Could not find AI text content in agent_result")
        raise ValueError("No AI response text found to email")

    logger.info("Agent response text extracted successfully")

    # Build SMTP2GO request payload
    payload = {
        "sender": sender_email,
        "to": [recipient_email],
        "subject": subject,
        "html_body": latest_text,
    }

    logger.info("SMTP2GO payload created")

    # Send POST request to SMTP2GO Standard Email endpoint
    url = "https://api.smtp2go.com/v3/email/send"
    data = json.dumps(payload).encode("utf-8")

    request = urllib.request.Request(
        url=url,
        data=data,
        method="POST",
        headers={
            "Content-Type": "application/json",
            "accept": "application/json",
            "X-Smtp2go-Api-Key": api_key,
        },
    )

    try:
        logger.info("Sending request to SMTP2GO API")
        with urllib.request.urlopen(request, timeout=30) as response:
            response_body = response.read().decode("utf-8")
            response_data = json.loads(response_body)

        logger.info(f"SMTP2GO response received with HTTP {response.status}")
        logger.info(f"SMTP2GO response data: {response_data}")
        return response_data

    except urllib.error.HTTPError as error:
        error_body = error.read().decode("utf-8", errors="replace")
        logger.error(f"SMTP2GO HTTPError {error.code}: {error_body}")
        raise RuntimeError(f"SMTP2GO request failed with status {error.code}") from error

    except urllib.error.URLError as error:
        logger.error(f"SMTP2GO URLError: {error}")
        raise RuntimeError(f"SMTP2GO connection failed: {error}") from error

In [47]:
send_agent_response_via_smtp2go(result)

2026-06-03 16:33:32.413 | INFO     | __main__:send_agent_response_via_smtp2go:3 - Starting SMTP2GO send process
2026-06-03 16:33:32.414 | INFO     | __main__:send_agent_response_via_smtp2go:21 - SMTP2GO environment variables loaded successfully
2026-06-03 16:33:32.415 | INFO     | __main__:send_agent_response_via_smtp2go:49 - Agent response text extracted successfully
2026-06-03 16:33:32.416 | INFO     | __main__:send_agent_response_via_smtp2go:59 - SMTP2GO payload created
2026-06-03 16:33:32.417 | INFO     | __main__:send_agent_response_via_smtp2go:77 - Sending request to SMTP2GO API
2026-06-03 16:33:34.243 | INFO     | __main__:send_agent_response_via_smtp2go:82 - SMTP2GO response received with HTTP 200
2026-06-03 16:33:34.243 | INFO     | __main__:send_agent_response_via_smtp2go:83 - SMTP2GO response data: {'request_id': '6f2477c4-d71f-459e-b37f-b362c6bb3a80', 'data': {'succeeded': 1, 'failed': 0, 'failures': [], 'email_id': '1wUoX8-FnQW0hPzjMM-cd6i'}}


{'request_id': '6f2477c4-d71f-459e-b37f-b362c6bb3a80',
 'data': {'succeeded': 1,
  'failed': 0,
  'failures': [],
  'email_id': '1wUoX8-FnQW0hPzjMM-cd6i'}}